On DAP, we can't install ipykernel in `.venv`. Thus, we need to apply a different strategy.

* do `unzip -o graph-ensemble.zip` 
If not working, remove `mc rm -rf graph-ensemble` and, then, `unzip graph-ensembles.zip`;
* `deactivate` (the `virtual-env`) such that the standard jupter kernel is selected;
* install via `pip install --editable . --config-settings editable_mode=compat`
* create a cell down here and do `import graph_ensembles` to check for the installation;

Usefull Commands:
* For this project, ```vars``` and ```plots``` are saved in `./outputs`;
* Delete folders on dap: ```mc rm --force --purge folder```
* To save new plots, delete previous ones with
```mc rm --force --purge dap/corealgos/rmilocco/outputs/datasets/ING-Directed```

On local:
* Linux: Zip outside the ``graph-ensembles`` folder: 
<br>```zip -rX graph-ensembles.zip graph-ensembles -x "graph-ensembles/src/graph_ensembles.egg-info/*" ".*" "*/.*" "*/__pycache__/*" "*/sythetic_network/*"```

Run the right install cmd from [pytorch](https://pytorch.org/), based on your architecture

Claim: by reconstructing the unobserved, we may close the gap between the total Page-Rank and Page-Rank only inside the ING-clients

1) Split the nodes into ING (`vI`) and `ROW` (`vR`);

2) Find ``vI`` and ``eI`` as the edges only between. We will cal `intra` (`eI`) edges, `bet` (ING-ROW), `row` (non ING interacting clients); 

4) Calculate the Page-Rank of only the `intra` nodes and compare it with the full graph PR.
Now, we expect that the 2 PR are different. So, help this bias by reconstructing the missing part. Ref [LateX](https://asajadi.github.io/fast-pagerank/) based on [MathWorks](https://www.mathworks.com/content/dam/mathworks/mathworks-dot-com/moler/exm/chapters/pagerank.pdf);

5) Calculate the strengths taking into account also the ING-ROW fluxes, while discarding the self-payments;
6) Freeze the `eI` and fit $\delta$ parameter as 
    * $L_I \stackrel{!}{=} \langle L_I \rangle(\delta_I) := \sum_{i \in I, j \in I} p_{ij}(\delta_I)$;

    * $L^{no-I}_U = L_I + L_{bet} \stackrel{!}{=} L_I + \sum_{i \in I, r \in R} (p_{ir}(\delta_{U}) + p_{ri}(\delta_{U})) $;

    * $L_U = L_I + L_{bet} \stackrel{!}{=} \sum_{(i,j) \in \left\{(I,I),(I,R),(R,I) \right\} } p_{ij}(\delta_{U}) $;

7) Page-Rank (or Influence Vector);

In [ ]:
# auto-reload the packages at every run
%load_ext autoreload
%autoreload 2

#display all the results not only the last one
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import os
try:
    corpkey = True if os.environ['DSBOX_USERNAME'] else None
    # %pip install matplotlib pandas scipy tqdm torch torchvision
except:
    corpkey = None

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import graph_ensembles as ge
from graph_ensembles import sparse as sp
import graph_ensembles.utils as utils
import graph_ensembles.dependencies as dep
from graph_ensembles.plots import plotting_functions as plot

# plot.plot_local_fonts(corpkey)
utils._set_mpl_params(fontsize=20)
utils.check_cpu_gpu_with_torch()

dataset_name = "ING" #"recNET"
dataset_direction = "Directed"
id_code, cg_method, year = "grid_id", "random", 2022

In [ ]:
# note that inside the kwargs there is a copy of the pdtrans
max_num_entries = None #if corpkey else 3e3 #30e6 # old 1e3
pdtrans, kwargs, total_levels = \
    ge.dataset_loader(dataset_name, dataset_direction = dataset_direction,
                    corpkey = corpkey, id_code = id_code, cg_method = cg_method,
                    year = year, max_num_entries = max_num_entries) #63332573

pdtrans.info()

Define all the vertex and edges

In [ ]:
e = pdtrans.loc[:, [f'payer_{id_code}', f'beneficiary_{id_code}']]
unique_nodes_from = lambda df: pd.DataFrame(data = np.unique(df.to_numpy().ravel('K')), columns = ["id"])
v = unique_nodes_from(e) # = 1 column DataFrame with "id" the node label, whereas the index is the incresing integer node-index
e = pdtrans.iloc[:, ::2] # payer, beneficiary, amount_euro
e.columns = ["src", "dst", "amount"] 

e.head()
print(f'-v.shape: {v.shape}',)

Create Full Graph

In [4]:
pr_name = "page_rank"
kwargs_graph = {'name' : 'ING', 'level' : 0, 'corpkey' : corpkey, "_pr_name" : pr_name}
g = sp.graphs.DiGraph(v, e, **kwargs_graph)
# g.load_or_create_degrees()

# del pdtrans

In [5]:
g._kwargs_pr = {"p" : 0.85, "max_iter" : 200, "tol" : 1e-06}
g._pr = g.pagerank_power(**g._kwargs_pr)
# g.save_vars()

Create a split of ``v`` into `vI`

Fit the `delta_intra`, for a splitting

In [ ]:
from tqdm import trange
from itertools import product

vsplits, intra_sizes = [62], sorted([0.2, .5, .8])

x0 = [1.8996372e-17]
num_sigmas = 2

# sample arguments
num_graph_samples_per_vsplit = 70
chunk_row_size = 2000
measures = ["_pr", "_out_degree", "_in_degree",]
rtol = 1e-8
fit_methods = ["num_edges_" + i for i in ["intra","intra_bet"]] #
g._topN_max_rank = 100

for intra_size_vsplit in product(intra_sizes, vsplits):
    intra_size, vsplit = intra_size_vsplit

    gI, vI, eI, idx_intra_nodes, unsampled_vI, frozen_edges = g.vsplit_intra_and_calculate_measures(v, e, intra_size, vsplit, kwargs_graph, measures)

    # create the model kwargs
    kwargs_model = kwargs_graph.copy()
    kwargs_model.update({"name" : "MultiScaleMod", "intra_size" : intra_size,
                        "prop_out_I" : g.out_strength(), "prop_in_I" : g.in_strength(),
                        "prop_out_R" : np.array([0]), "prop_in_R" : np.array([0]),
                        "num_sigmas" : num_sigmas
                        })

    # since intra_sizes are sorted, this line changes only the last iteration
    fit_methods = ["num_edges_intra"] if intra_size == 1 else fit_methods 
    
    for fit_method in fit_methods:
        print(f'\n-Dealing with intra_size, vsplit, fit_method {intra_size, vsplit, fit_method}',)

        # vR and num_edges_bet are needed for the fitting procedure
        vR, num_edges = g.vsplit_row(v, vI, e, idx_intra_nodes, gI.num_edges(), fit_method)
        
        # update the model kwarg with the current fit_method
        kwargs_model.update({"fit_method" : fit_method, "num_edges" : num_edges})

        # create the model and fit it
        model = sp.ScaleInvariantModel.initialize_model(g, gI, vR, kwargs_model)
        num_start_graph = model.set_ensemble_variables(measures, num_graph_samples_per_vsplit)
        model.load_or_fit(x0 = x0[0], maxiter = 30, verbose = 0)

        # if intra_size < 1, set the right strengths to predict the full-network
        if intra_size < 1: model.set_observables_from(g)

        print(f'\n-For vsplitting vsplit {vsplit}: {num_start_graph} graphs already sampled, {np.clip(num_graph_samples_per_vsplit - num_start_graph, 0, None)} remaining')

        # send variables (model.params, ...) to cuda:0
        unsampled_vI = model.send_variables_to_gpu(unsampled_vI)

        # def var for model vars
        mod_vars = model.__dict__
        
        # Note: the seed = vsplit only works for solo-agent. The graph-sampling is parallelized, so it would be random even if vsplit specified.
        # That's why .sample() has graph_idx as argument
        # set graph_idx = 0, since if all the graphs are already sampled in the next calculations it will have a number different to None
        for graph_idx in trange(num_start_graph, num_graph_samples_per_vsplit, 
                        desc=f"-Total progress {int(np.round((vsplit+1)/len(vsplits) * 100))}%, Inner Graph Sampling", position = 0, leave= True):

            # sample
            gs = model.sample(ref_g = g, unsampled_vI = unsampled_vI, frozen_edges = frozen_edges, 
                            graph_idx = graph_idx, chunk_row_size = chunk_row_size)
            gs.calculate_measures(g, measures)
            gs.set_pr_on_I(gI)
            
            # save the ensemble average and std for every measures on the model class
            for m in measures:
                if num_start_graph == graph_idx == 0:
                    mod_vars[f"prev{m}"], mod_vars[f"prev{m}_std"] = 0, 0
                mod_vars[f"prev{m}"], mod_vars[f"prev{m}_std"] = \
                        model.recursive_mean_std(graph_idx, mod_vars[f"prev{m}"], mod_vars[f"prev{m}_std"], gs.__dict__[m])

        for m in measures:
            mod_vars[m] = mod_vars[f"prev{m}"]
            mod_vars[m+"_std"] = mod_vars[f"prev{m}_std"]

        # set pr on I and rescale the internal one to much the fraction of time spent there
        utils.set_model_pr_on_I(model, gI)
        gI.rescale_pr_with(model, scaler = True)
        num_bins = int(np.sqrt(model.num_vertices))
        plot.pr_on_internal_nodes(model, g, gI, num_bins)
        plot.pr_on_internal_nodes_vs_rank(model, g, gI)
            
        # compute the expected degree and plt them
        unsampled_vI = model.send_variables_to_cpu(unsampled_vI)
        _ = model.expected_degree(unsampled_vI, gI)
        plot.ccdf_deg_out_in(g, gI, model)

        # calculate the deg/stre on I and overlaps
        g.calculate_out_in_degree_strength_on_I(gI, model)
        g.topN_overlap_pr_deg_stre(gI, model)
        
        # calculate only the overlap between pr
        g.topN_overlap_pr(gI)
        g.topN_overlap_pr(model)

        # plot the overlaps
        plot.topN_overlap_pr_deg_stre(g, gI, model)
        plot.topN_overlap_pr(gI, model)